# How the graph is built: criterion x quantile

The dial notebooks move what the receptor vector is *mixed from*. This one moves
**which edges exist at all**. The message-passing graph keeps only K train molecules:
the quantile sets how many survive, the criterion picks which ones
(`orbind/mol_selection.py`).

Every number the paper reports stands on ONE point of this grid. The figures below
exist to say whether that point is defensible, not merely inherited -- so the point
itself is marked on every panel.

**This notebook draws and does not aggregate.** Fold means, intervals and paired
differences all come from `scripts/article_sweeps/s4_quantile_grid.py`, which delegates
the averaging rule to `alpha_grid`: model seeds are averaged *inside* each fold, and
the interval is over folds. A figure that re-derived its own means would treat the
(fold, seed) cells as independent observations and halve every band on the page.

Produced by `scripts/article_sweeps/s4_run_quantile_criteria.py`.

In [ ]:
import pathlib
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "notebooks" / "article_figures"))

import figkit as fk
from scripts.article_sweeps import s4_quantile_grid as qg

fk.use_house_style()
print("repo:", ROOT)

## Knobs

In [ ]:
# ================================== KNOBS ====================================
ROOT_DIR   = "results/article_sweeps/quantile_criteria"
DATASET    = None          # None -> everything on disk; or "m2or" / ["cc", "hc"]
REGIME     = None          # None -> both; or "transductive" / "inductive"
MOL_SOURCE = None          # None -> whatever is there; or "chemberta"
COMBO      = "cls+mol"     # the head; the reference arm is kept whatever this says
SPLIT      = "test"        # "val" is the honest place to CHOOSE a quantile

CI_LEVEL   = 0.95          # Student-t over FOLDS (seeds averaged inside each fold)
METRICS    = None          # None -> the metric of record per dataset
HIGHLIGHT  = "auto"        # "auto" -> the criterion this dataset reports; None -> all
                           # seven in full colour; or name one criterion
SHOW_K     = True          # the K panel: what the quantile actually cuts

SAVE_FIGS  = False
FIG_DIR    = "results/article_figures/quantile_criteria"
# =============================================================================

df = qg.load(root=ROOT_DIR, dataset=DATASET, regime=REGIME, mol_source=MOL_SOURCE,
             combo=COMBO, split=SPLIT)

SERIES = sorted(df.series.unique())
QS = sorted(pd.to_numeric(qg.graph_rows(df)["quantile"], errors="coerce").dropna().unique())
CRITS = [c for c in fk.CRIT if c in set(qg.graph_rows(df).criterion)]

In [ ]:
# ---------------------------------------------------------------- what is on disk
# One guard, and it matters: `k_mode` decides what a quantile MEANS. Under
# `coverage_quantile` q cuts on the coverage distribution; under `fraction` it keeps
# the top (1-q) share outright. On a complete matrix the first keeps everything at
# every q. A frame holding both puts two different knobs on one x axis.
modes = sorted(df.k_mode.astype(str).unique())
if len(modes) != 1:
    raise SystemExit(f"this frame holds k_mode {modes} -- two knobs cannot share an "
                     f"axis. Filter to one, or read them as separate figures.")
K_MODE = modes[0]

print(f"{len(SERIES)} series | {len(QS)} quantiles | {len(CRITS)} criteria | "
      f"k_mode: {K_MODE} | split: {SPLIT}")
print("quantiles:", ", ".join(f"{q:g}" for q in QS))
print("files:", ", ".join(pathlib.Path(f).name for f in df.attrs["files"]))
qg.coverage(df)

In [ ]:
MOF = qg.metric_for(df)
DS_OF = dict(zip(df.series, df.dataset))
print("metric of record:", MOF)

# How small a difference this grid can resolve at all: the model noise LEFT IN A FOLD
# MEAN after the seeds are averaged. Every curve below is made of fold means, so a
# wiggle shorter than this is not a finding, whatever the band says. With one seed it
# is undefined -- and then no I-mark is drawn, which is the honest statement.
FLOOR = {}
for s in SERIES:
    f = qg.floor(df[df.series == s], MOF[DS_OF[s]])
    if len(f):
        FLOOR[s] = float(f.iloc[0]["floor"])
if FLOOR:
    display(pd.Series(FLOOR, name="resolution floor").round(4))
else:
    print("one model seed per cell -- no resolution floor; re-run with --seeds 42 43 ...")

---
## 1. The quantile

One line per criterion, the boosting reference as a grey dashed level, and a ring on
the cell the paper reports.

In [ ]:
# 1.1  THE METRIC OF RECORD AGAINST THE QUANTILE.
# Dimming the six criteria we do not report makes the panel readable, but only if the
# LEGEND is dimmed with it -- a key that shows seven hues beside a plot drawn in grey
# is worse than no key. So the highlight is figure-wide: when the panels disagree about
# which criterion is theirs (m2or reports greedy_pair_cover, the insects report
# coverage), nothing is dimmed and all seven keep their own colour.
def highlight_for(series):
    if HIGHLIGHT != "auto":
        return HIGHLIGHT
    return qg.PAPER_POINT.get(DS_OF[series], (None, None))[0]

_wanted = {highlight_for(s) for s in SERIES}
HL = _wanted.pop() if len(_wanted) == 1 else None
if HL is None and len(_wanted) == 0:
    print("no criterion highlighted")
else:
    print("highlighted:", HL or "none (the panels report different criteria)")

fig, ax = fk.panels(len(SERIES), ncols=min(3, len(SERIES)), w=3.4, h=2.6)
for i, s in enumerate(SERIES):
    a, metric = ax[i], MOF[DS_OF[s]]
    sub = df[df.series == s]
    c = qg.curve(sub, metric, level=CI_LEVEL)
    drawn = False
    for crit in CRITS:
        q = c[c.criterion == crit].sort_values("quantile")
        if q.empty:
            continue
        col, mk, lw, z = fk.crit_style(crit, HL)
        fk.band(a, q["quantile"], q["lo"], q["hi"], col, 0.10 if crit == HL else 0.0)
        a.plot(q["quantile"], q["mean"], color=col, marker=mk, lw=lw, zorder=z,
               markersize=4.5 if crit == HL else 3.2)
        drawn = True
    lv = qg.levels(sub, metric, level=CI_LEVEL)
    if len(lv):
        fk.reference_line(a, lv.iloc[0]["mean"], "boost_full")
    # the cell the paper stands on, ringed rather than annotated: the label would
    # collide with six other lines on a panel this size
    pp = qg.paper_point(sub, metric, level=CI_LEVEL)
    if len(pp) and bool(pp.iloc[0]["in_grid"]):
        a.plot([pp.iloc[0]["quantile"]], [pp.iloc[0]["mean"]], marker="o",
               markersize=11, markerfacecolor="none", markeredgecolor=fk.INK,
               markeredgewidth=1.2, zorder=4)
    fk.knob_axis(a, QS, label=f"quantile ({K_MODE})")
    if drawn:
        fk.resolution_mark(a, FLOOR.get(s))
    else:
        fk.note_empty(a)
    a.set_title(s, fontsize=8.5)
    a.set_ylabel(metric)

fig.suptitle("Graph construction: which molecules carry the messages", y=1.005,
             fontsize=11)
handles = []
for c in CRITS:
    col, mk, lw, _z = fk.crit_style(c, HL)
    handles.append(Line2D([], [], color=col, marker=mk, lw=lw,
                          label=c + ("  (reported)" if c == HL else "")))
handles += [Line2D([], [], color=fk.C["boost_full"], ls=fk.DASH["boost_full"],
                   label="boost [prot || mol]"),
            Line2D([], [], color=fk.INK, marker="o", ls="none", markerfacecolor="none",
                   markersize=9, label="the cell the paper reports")]
fk.figlegend(fig, handles, ncol=3, pad_in=1.0)
fk.savefig(fig, "quantile_metric", FIG_DIR, SAVE_FIGS)
plt.show()

### 1.2 The advantage over the boosting base, paired

The panel above shows two absolute levels. This one shows their difference computed
**fold by fold**, which is the comparison the paper makes and the only one whose
interval means what it appears to mean. Zero is the base.

The difference stays in the metric's own units, so on RMSE and MAE a curve *below*
zero is the graph winning.

In [ ]:
# 1.2  PAIRED ADVANTAGE OVER THE BASE.
fig, ax = fk.panels(len(SERIES), ncols=min(3, len(SERIES)), w=3.4, h=2.6)
for i, s in enumerate(SERIES):
    a, metric = ax[i], MOF[DS_OF[s]]
    d = qg.delta_vs_boost(df[df.series == s], metric, level=CI_LEVEL)
    a.axhline(0.0, color=fk.C["boost_full"], lw=1.3, ls=fk.DASH["boost_full"])
    for crit in CRITS:
        q = d[d.criterion == crit].sort_values("quantile")
        if q.empty:
            continue
        col, mk, lw, z = fk.crit_style(crit, HL)
        fk.band(a, q["quantile"], q["lo"], q["hi"], col, 0.10 if crit == HL else 0.0)
        a.plot(q["quantile"], q["mean"], color=col, marker=mk, lw=lw, zorder=z)
    fk.knob_axis(a, QS, label=f"quantile ({K_MODE})")
    if d.empty:
        fk.note_empty(a)
    a.set_title(s, fontsize=8.5)
    a.set_ylabel(f"{metric} - base")

fig.suptitle("Paired advantage over the boosting base, fold by fold", y=1.005,
             fontsize=11)
fk.figlegend(fig, handles[:-1], ncol=3, pad_in=1.0)
fk.savefig(fig, "quantile_delta", FIG_DIR, SAVE_FIGS)
plt.show()

### 1.3 What the quantile actually cuts

K, not q, is the quantity the knob controls. Its own panel and never a second y axis
on the plot above: two quantiles that resolve to the same K are one experiment drawn
twice, and this is where that becomes visible.

In [ ]:
# 1.3  K AGAINST THE QUANTILE.
if SHOW_K:
    k = qg.k_curve(df)
    fig, ax = fk.panels(len(SERIES), ncols=min(3, len(SERIES)), w=3.4, h=2.2)
    for i, s in enumerate(SERIES):
        a = ax[i]
        q = k[k.series == s].sort_values("quantile")
        if q.empty:
            fk.note_empty(a)
        else:
            a.plot(q["quantile"], q["K"], color=fk.MUTED, marker="o")
            fk.band(a, q["quantile"], q["K_min"], q["K_max"], fk.MUTED, 0.18)
            if (q["K"].nunique() == 1):
                a.text(0.5, 0.9, "the knob cuts nothing here", ha="center",
                       transform=a.transAxes, fontsize=8, color=fk.C["fun"])
        fk.knob_axis(a, QS, label=f"quantile ({K_MODE})")
        a.set_title(s, fontsize=8.5)
        a.set_ylabel("molecules kept (K)")
    fig.suptitle("How many train molecules survive the cut", y=1.01, fontsize=11)
    fk.figlegend(fig, [Line2D([], [], color=fk.MUTED, marker="o",
                              label="K (band = fold to fold)")], ncol=1)
    fk.savefig(fig, "quantile_K", FIG_DIR, SAVE_FIGS)
    plt.show()

---
## 2. Where each criterion peaks, and whether the peak is real

`sep` is the gap between the best quantile and the runner-up, measured in half-widths
of the best point's own interval. **Below 1 the optimum is a ranking of noise** -- and
"the knob does not matter over this range" is a result, and a far easier one to defend
than a peak that moves with the seed.

`behind` in the second table is how far the cell the paper reports sits below the best
cell of the same series, in that cell's half-widths.

In [ ]:
for s in SERIES:
    metric = MOF[DS_OF[s]]
    sub = df[df.series == s]
    print(f"\n=== {s}   ({metric}, {K_MODE})")
    display(qg.best_q(sub, metric, level=CI_LEVEL).drop(columns="series").round(4))
    pp = qg.paper_point(sub, metric, level=CI_LEVEL)
    if len(pp):
        display(pp.drop(columns="series").round(4))

---
## 3. The whole battery

A construction that wins on one metric and nowhere else has not won.

In [ ]:
for s in SERIES:
    sub = df[df.series == s]
    metrics = METRICS or qg.metrics_available(sub, dataset=DS_OF[s], which="headline")
    have = [m for m in metrics
            if pd.to_numeric(sub[m], errors="coerce").notna().any()]
    if not have:
        continue
    fig, ax = fk.panels(len(have), ncols=min(4, len(have)), w=2.8, h=2.2)
    for j, m in enumerate(have):
        a = ax[j]
        c = qg.curve(sub, m, level=CI_LEVEL)
        for crit in CRITS:
            q = c[c.criterion == crit].sort_values("quantile")
            if q.empty:
                continue
            col, mk, lw, z = fk.crit_style(crit, HL)
            a.plot(q["quantile"], q["mean"], color=col, marker=mk, lw=lw, zorder=z)
        lv = qg.levels(sub, m, level=CI_LEVEL)
        if len(lv):
            fk.reference_line(a, lv.iloc[0]["mean"], "boost_full")
        fk.knob_axis(a, QS)
        a.set_title(m + ("  (lower is better)" if m in qg.LOWER_IS_BETTER else ""),
                    fontsize=8)
    fig.suptitle(s, y=1.01, fontsize=10.5)
    fk.figlegend(fig, handles[:-1], ncol=4, pad_in=1.0)
    fk.savefig(fig, f"quantile_battery_{s.replace(' / ', '_')}", FIG_DIR, SAVE_FIGS)
    plt.show()

---
## Before quoting a quantile from this page

Reading the best q off these curves and then reporting those same curves chooses the
number and its defence from the same rows. Two honest forms:

* run the sweep with `SPLIT = "val"`, pick there, and read test once;
* or make the weaker claim these figures support directly -- that over the range where
  `sep < 1` the construction does not matter, and the cell we report is not behind the
  best one by more than the grid can resolve.

The second is usually the one worth making. The ablation's job is to show the choice
was not load-bearing, not to crown a winner.